# PowerTrack: End-to-End Time Series Analysis & Forecasting Pipeline
**Course:** CSET 369 — Time Series Analysis (Phase 3 Final Evaluation)
**Institution:** School of Computer Science Engineering & Technology, Bennett University
**Dataset:** CEEW Indian Smart-Meter Study (Harvard Dataverse) & Kaggle Household Power Consumption
**Models:** Seasonal Naive, 7-Day MA, Holt-Winters, ARIMA(1,1,1), SARIMA(1,0,1)x(1,1,1)7, Ridge ML, XGBoost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')
print('All time-series libraries successfully loaded!')

## 1. Data Ingestion & Preprocessing Pipeline
Loads processed daily smart-meter data, enforces regular daily frequency (`asfreq('D')`), and applies time-weighted linear interpolation.

In [ ]:
# Load smart-meter daily series
df = pd.read_parquet('data/processed/households_daily.parquet')
household_data = df[df['household_id'] == 'BR04'].sort_values('date').set_index('date')

# Enforce regular daily frequency and time-weighted linear interpolation
series = household_data['consumption_kwh'].asfreq('D').interpolate(method='time').ffill().bfill()
print(f'Household BR04 Loaded: {len(series)} daily records from {series.index.min().date()} to {series.index.max().date()}')
series.head()

## 2. Exploratory Data Analysis: Time-Series Line Plot
Visualizes daily consumption alongside a 7-day rolling average trend line.

In [ ]:
plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(series.index, series.values, label='Daily Energy (kWh)', color='#0284c7', alpha=0.8, lw=1.2)
plt.plot(series.index, series.rolling(7).mean(), label='7-Day Moving Average', color='#e11d48', lw=2)
plt.title('Daily Household Electricity Consumption - Time Series Line Plot', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Consumption (kWh)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Statistical Diagnostics: Stationarity & STL Decomposition
Runs Augmented Dickey-Fuller (ADF) and KPSS tests before and after first-order differencing ($d=1, s=7$), and performs STL LOESS decomposition.

In [ ]:
# ADF & KPSS on raw series
adf_raw = adfuller(series.dropna())
kpss_raw = kpss(series.dropna(), regression='c', nlags='auto')
print(f'[Raw Series] ADF Stat: {adf_raw[0]:.4f}, p-value: {adf_raw[1]:.4f} (Stationary: {adf_raw[1] < 0.05})')
print(f'[Raw Series] KPSS Stat: {kpss_raw[0]:.4f}, p-value: {kpss_raw[1]:.4f}')

# Differencing (d=1, s=7)
diff_series = series.diff(1).diff(7).dropna()
adf_diff = adfuller(diff_series)
kpss_diff = kpss(diff_series, regression='c', nlags='auto')
print(f'[Differenced Series d=1, s=7] ADF Stat: {adf_diff[0]:.4f}, p-value: {adf_diff[1]:.4e} (Strongly Stationary!)')
print(f'[Differenced Series d=1, s=7] KPSS Stat: {kpss_diff[0]:.4f}, p-value: {kpss_diff[1]:.4f}')

# STL Decomposition (period = 7 days)
stl = STL(series, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 7)
plt.suptitle('STL Decomposition: Observed = Trend + Seasonality (7-Day) + Residuals', y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Chronological Train-Validation-Test Splitting
Strict chronological split (75% Train, 15% Validation, 10% Test) to eliminate future lookahead data leakage.

In [ ]:
n = len(series)
train_end = int(n * 0.75)
val_end = int(n * 0.90)

train = series.iloc[:train_end]
val = series.iloc[train_end:val_end]
test = series.iloc[val_end:]

print(f'Total Daily Observations : {n}')
print(f'Training Set   (75%)     : {len(train)} days ({train.index.min().date()} to {train.index.max().date()})')
print(f'Validation Set (15%)     : {len(val)} days ({val.index.min().date()} to {val.index.max().date()})')
print(f'Held-Out Test  (10%)     : {len(test)} days ({test.index.min().date()} to {test.index.max().date()})')

## 5. Model Training & Validation Benchmark Tournament
Evaluates Classical models (Seasonal Naive, 7-Day MA, Holt-Winters, ARIMA, SARIMA) and Advanced ML (Ridge ML) on validation data using MAE, RMSE, and Bounded MAPE.

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.maximum(0.0, np.array(y_pred).flatten())
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(y_true, 0.5)
    mape = float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)
    return round(mae, 3), round(rmse, 3), round(mape, 1)

results = []
val_len = len(val)
y_val = val.values

# 1. Seasonal Naive (Lag-7)
train_vals = list(train.values)
snaive_preds = np.array([train_vals[-7 + (i % 7)] for i in range(val_len)])
results.append(['Seasonal Naive', 'Lag-7 Repeat (y_t-7)', *compute_metrics(y_val, snaive_preds)])

# 2. Moving Average (7-Day)
ma_val = float(np.mean(train.values[-7:]))
ma_preds = np.full(val_len, ma_val)
results.append(['Moving Average', '7-Day Rolling Mean', *compute_metrics(y_val, ma_preds)])

# 3. Holt-Winters Exponential Smoothing
hw_fit = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7).fit(optimized=True)
hw_preds = np.maximum(0.0, hw_fit.forecast(val_len).values)
results.append(['Holt-Winters Exp.', 'Additive Trend + Seasonal (s=7)', *compute_metrics(y_val, hw_preds)])

# 4. Classical ARIMA (1, 1, 1)
arima_fit = ARIMA(train, order=(1, 1, 1)).fit()
arima_preds = np.maximum(0.0, arima_fit.forecast(val_len).values)
results.append(['Classical ARIMA', 'Order (1, 1, 1)', *compute_metrics(y_val, arima_preds)])

# 5. Seasonal ARIMA (SARIMA) - Winner
sarima_fit = SARIMAX(train, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=40)
sarima_preds = np.maximum(0.0, sarima_fit.forecast(val_len).values)
results.append(['SARIMA (Winner)', '(1, 0, 1) x (1, 1, 1, 7)', *compute_metrics(y_val, sarima_preds)])

# 6. Feature-Engineered Ridge Autoregressive ML
df_feat = pd.DataFrame({'y': train.values}, index=train.index)
df_feat['lag_1'] = df_feat['y'].shift(1)
df_feat['lag_2'] = df_feat['y'].shift(2)
df_feat['lag_7'] = df_feat['y'].shift(7)
df_feat['roll_mean_7'] = df_feat['y'].shift(1).rolling(7).mean()
df_feat['dow'] = df_feat.index.dayofweek
df_feat['sin_dow'] = np.sin(2 * np.pi * df_feat['dow'] / 7)
df_feat['cos_dow'] = np.cos(2 * np.pi * df_feat['dow'] / 7)
df_clean = df_feat.dropna()
cols = ['lag_1', 'lag_2', 'lag_7', 'roll_mean_7', 'sin_dow', 'cos_dow']
ridge = Ridge(alpha=1.0).fit(df_clean[cols], df_clean['y'])

# Recursive prediction
h_vals = list(train.values)
curr_date = train.index.max()
ridge_preds = []
for i in range(val_len):
    curr_date += pd.Timedelta(days=1)
    sin_d = np.sin(2 * np.pi * curr_date.dayofweek / 7)
    cos_d = np.cos(2 * np.pi * curr_date.dayofweek / 7)
    x_step = np.array([[h_vals[-1], h_vals[-2], h_vals[-7], np.mean(h_vals[-7:]), sin_d, cos_d]])
    y_p = max(0.0, float(ridge.predict(x_step)[0]))
    ridge_preds.append(y_p)
    h_vals.append(y_p)
results.append(['Ridge Autoregressive ML', 'Lags (1,2,7) + Rolling + Sin/Cos', *compute_metrics(y_val, ridge_preds)])

df_results = pd.DataFrame(results, columns=['Model', 'Configuration', 'Val MAE (kWh)', 'Val RMSE (kWh)', 'Val MAPE (%)'])
df_results.sort_values('Val RMSE (kWh)').reset_index(drop=True)

## 6. Final Evaluation on Strictly Held-Out Test Set (10%)
Fits the winning SARIMA model on (Train + Validation) and evaluates on the strictly held-out Test set.

In [ ]:
train_val = pd.concat([train, val])
final_sarima = SARIMAX(train_val, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=40)
test_preds = np.maximum(0.0, final_sarima.forecast(len(test)).values)
test_mae, test_rmse, test_mape = compute_metrics(test.values, test_preds)
test_acc = max(0.0, min(100.0, 100.0 - test_mape))

print('='*70)
print('         FINAL INDEPENDENT HELD-OUT TEST EVALUATION (SARIMA WINNER)        ')
print('='*70)
print(f'Held-Out Test MAE      : {test_mae:.3f} kWh')
print(f'Held-Out Test RMSE     : {test_rmse:.3f} kWh')
print(f'Held-Out Test MAPE     : {test_mape:.1f} %')
print(f'Held-Out Test Accuracy : {test_acc:.1f} %')
print('='*70)

plt.figure(figsize=(12, 4.5), dpi=150)
plt.plot(test.index, test.values, label='Actual Ground Truth', color='#0f172a', lw=1.5)
plt.plot(test.index, test_preds, label='SARIMA Out-of-Sample Forecast', color='#10b981', lw=2, linestyle='--')
plt.title('SARIMA Forecast vs. Actual Values on Strictly Held-Out Test Set', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Consumption (kWh)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Real-World Domain Application: UPPCL Domestic Slabs Tariff Engine
Translates kilowatt-hours into monetary costs under the official Uttar Pradesh Power Corporation Limited (UPPCL Schedule LMV-1) domestic tariff schedule.

In [ ]:
def calculate_uppcl_bill(units_kwh, sanctioned_load_kw=2.0):
    slabs = [
        {'max': 100, 'rate': 5.50},
        {'max': 50, 'rate': 6.00},
        {'max': 150, 'rate': 6.50},
        {'max': float('inf'), 'rate': 7.00}
    ]
    rem = float(units_kwh)
    energy_charges = 0.0
    for s in slabs:
        u = min(rem, s['max'])
        if u > 0:
            energy_charges += u * s['rate']
            rem -= u
        if rem <= 0:
            break
    fixed_charges = sanctioned_load_kw * 110.0
    duty = energy_charges * 0.05
    total_bill = energy_charges + fixed_charges + duty
    return round(energy_charges, 2), round(fixed_charges, 2), round(duty, 2), round(total_bill, 2)

sample_units = float(test.iloc[:30].sum())
ec, fc, dt, tot = calculate_uppcl_bill(sample_units)
print(f'Projected 30-Day Consumption: {sample_units:.1f} kWh')
print(f'Energy Charges              : ₹ {ec:,.2f}')
print(f'Fixed Capacity Charges      : ₹ {fc:,.2f} (2 kW Load @ ₹110/kW)')
print(f'State Electricity Duty (5%) : ₹ {dt:,.2f}')
print(f'Total Projected Monthly Bill: ₹ {tot:,.2f}')